In [2]:
from pinecone import Pinecone, PodSpec
from pinecone import Pinecone, ServerlessSpec

import os, getpass

In [3]:
os.environ["PINECONE_API_KEY"] = getpass.getpass("Ingresa tu API Key de Pinecone : ")
api_key = os.environ["PINECONE_API_KEY"]

## Creando un index (tabla) en Pinecone

In [4]:
index_name = "knowledge-base"
dimension = 1536  # text-embedding-3-small

pc = Pinecone(api_key=api_key)

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

print(f"Index {index_name} creado con éxito en Pinecone")

Index knowledge-base creado con éxito en Pinecone


## Estadísticas de un index

In [5]:
pc.describe_index(index_name)

{
    "name": "knowledge-base",
    "metric": "cosine",
    "host": "knowledge-base-346svke.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 1536,
    "deletion_protection": "disabled",
    "tags": null
}

## Lista todos los index en Pinecone

In [6]:
pc.list_indexes()

[
    {
        "name": "knowledge-base",
        "metric": "cosine",
        "host": "knowledge-base-346svke.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "cloud": "aws",
                "region": "us-east-1"
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 1536,
        "deletion_protection": "disabled",
        "tags": null
    }
]

## Cargando datos (pdf) y generando sus fragmentos

In [7]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:

loader = PyPDFLoader("pdf/ebook-stable-difussion-guia.pdf")
data = loader.load()

text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    chunk_size = 400, 
    chunk_overlap = 20
)

docs = text_splitter.split_documents(documents = data)

In [9]:
len(docs), docs

(23,
 [Document(metadata={'producer': 'Adobe PDF library 17.00', 'creator': 'Adobe Illustrator 27.2 (Windows)', 'creationdate': '2023-01-28T16:11:35+02:00', 'moddate': '2023-01-28T16:27:11+01:00', 'title': 'Ebook Stable Difussion - Guia v4', 'source': 'pdf/ebook-stable-difussion-guia.pdf', 'total_pages': 24, 'page': 0, 'page_label': '1'}, page_content='ChatGPT + Stable Diffusion\nCómo crear Contenido con \nInteligencia Artificial'),
  Document(metadata={'producer': 'Adobe PDF library 17.00', 'creator': 'Adobe Illustrator 27.2 (Windows)', 'creationdate': '2023-01-28T16:11:35+02:00', 'moddate': '2023-01-28T16:27:11+01:00', 'title': 'Ebook Stable Difussion - Guia v4', 'source': 'pdf/ebook-stable-difussion-guia.pdf', 'total_pages': 24, 'page': 1, 'page_label': '2'}, page_content='Índice\nIntroducción\nCómo puede ayudarte la Inteligencia Artiﬁcial a crear contenido\nQué es Stable Diffusion\nQué es ChatGPT\n03\n03\n04\n05\nStable Diffusion 06\nGuía ChatGPT\nCómo puedes usar ChatGPT\nEjemplos

https://app.pinecone.io/

In [10]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("Ingresa tu API Key de OpenAI : ")

## Cargando embeddings a Pinecone

In [11]:
from langchain_pinecone import PineconeVectorStore
from langchain_openai import OpenAIEmbeddings

index_name = "knowledge-base"
embeddings = OpenAIEmbeddings()

docsearch = PineconeVectorStore.from_documents(
    documents=docs,
    embedding=embeddings,
    index_name=index_name
)

## Consultando datos del index

![Imagen](https://raw.githubusercontent.com/UKPLab/sentence-transformers/master/docs/img/SemanticSearch.png)

## Muestra los 5 embeddings más relevantes con la pregunta

In [ ]:
query = "¿qué es stable diffussion?"
docs = docsearch.similarity_search(query, k = 5) 
docs

In [ ]:
query = "¿en qué me puede ayudar la inteligencia artificial?"
docs = docsearch.similarity_search(query, k = 5)
docs

In [ ]:
print(docs[0].page_content)

## Eliminando index

In [12]:
pc = Pinecone(api_key=api_key)

pc.delete_index(index_name)